In [2]:
!pip install pgmpy

import numpy as np
import pandas as pd
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.models import BayesianNetwork          # (named DiscreteBayesianNetwork in pgmpy >= 1.0)
from pgmpy.inference import VariableElimination

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
heartDisease = pd.read_csv('heart.csv')
heartDisease = heartDisease.replace('?', np.nan).dropna()

In [4]:
print('Sample instances from the dataset are given below')
print(heartDisease.head())

Sample instances from the dataset are given below
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

  ca thal  heartdisease  
0  0    6             0  
1  3    3             2  
2  2    7             1  
3  0    3             0  
4  0    3             0  


In [5]:
print('\nAttributes and datatypes')
print(heartDisease.dtypes)


Attributes and datatypes
age               int64
sex               int64
cp                int64
trestbps          int64
chol              int64
fbs               int64
restecg           int64
thalach           int64
exang             int64
oldpeak         float64
slope             int64
ca               object
thal             object
heartdisease      int64
dtype: object


In [6]:
# Discretise continuous columns so the CPDs stay small
heartDisease['age']  = pd.cut(heartDisease['age'].astype(float),  [0, 40, 55, 70, 120], labels=[0, 1, 2, 3]).astype(int)
heartDisease['chol'] = pd.cut(heartDisease['chol'].astype(float), [0, 200, 240, 700],   labels=[0, 1, 2]).astype(int)

In [11]:
from pgmpy.models import DiscreteBayesianNetwork

# Model structure
model = DiscreteBayesianNetwork([('age', 'heartdisease'), ('sex', 'heartdisease'), ('exang', 'heartdisease'),
                                 ('cp', 'heartdisease'), ('heartdisease', 'restecg'), ('heartdisease', 'chol')])

print('\nLearning CPD using Maximum likelihood estimators')
model.fit(heartDisease)


Learning CPD using Maximum likelihood estimators


In [12]:
print('\nInferencing with Bayesian Network:')
infer = VariableElimination(model)


Inferencing with Bayesian Network:


In [13]:
print('\n1. Probability of HeartDisease given evidence= restecg')
print(infer.query(variables=['heartdisease'], evidence={'restecg': 1}))


1. Probability of HeartDisease given evidence= restecg
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(0) |              0.2223 |
+-----------------+---------------------+
| heartdisease(1) |              0.0000 |
+-----------------+---------------------+
| heartdisease(2) |              0.2509 |
+-----------------+---------------------+
| heartdisease(3) |              0.2317 |
+-----------------+---------------------+
| heartdisease(4) |              0.2951 |
+-----------------+---------------------+


In [14]:
print('\n2. Probability of HeartDisease given evidence= cp')
print(infer.query(variables=['heartdisease'], evidence={'cp': 2}))


2. Probability of HeartDisease given evidence= cp
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(0) |              0.6281 |
+-----------------+---------------------+
| heartdisease(1) |              0.1587 |
+-----------------+---------------------+
| heartdisease(2) |              0.0899 |
+-----------------+---------------------+
| heartdisease(3) |              0.0873 |
+-----------------+---------------------+
| heartdisease(4) |              0.0360 |
+-----------------+---------------------+
